# Notebook 02 — Data Cleaning & ETL Pipeline
**Project:** Olist E-Commerce Performance Analysis
**Purpose:** Clean all 9 tables, merge into one master dataset, engineer derived features
**Input:** Raw CSV files (never modified)
**Output:** data/processed/olist_master_cleaned.csv
**Reference:** docs/data_dictionary.md — Known Data Quality Issues

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
orders               = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items          = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
customers            = pd.read_csv("../data/raw/olist_customers_dataset.csv")
payments             = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews              = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
products             = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers              = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
geolocation          = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

print("All 9 raw files loaded — originals untouched")

All 9 raw files loaded — originals untouched


## Step 1 — Convert Datetime Columns
All 5 timestamp columns in the orders table are loaded as strings (object dtype).
Converting to datetime so we can compute delivery time and delay KPIs later.

In [3]:
datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col])

print("STEP 1 DONE: Datetime columns converted")
print(orders[datetime_cols].dtypes)

STEP 1 DONE: Datetime columns converted
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


## Step 2 — Filter to Delivered Orders Only
Non-delivered orders (shipped, canceled, unavailable etc.) have null delivery timestamps.
Keeping only 'delivered' orders to ensure clean KPI computation.
From extraction: 96,478 delivered out of 99,441 total orders.

In [4]:
before = len(orders)
orders_clean = orders[orders['order_status'] == 'delivered'].copy()
after = len(orders_clean)

print("STEP 2 DONE: Filtered to delivered orders only")
print(f"  Before : {before:,} rows")
print(f"  After  : {after:,} rows")
print(f"  Dropped: {before - after:,} non-delivered orders")

STEP 2 DONE: Filtered to delivered orders only
  Before : 99,441 rows
  After  : 96,478 rows
  Dropped: 2,963 non-delivered orders


## Step 3 — Clean Payments Table
Issues identified in extraction:
- 2 rows with payment_installments = 0 (invalid) → replace with 1
- 3 rows with payment_type = 'not_defined' → drop these rows
- Multiple payment rows per order → aggregate to one row per order

In [5]:
invalid_before = payments[payments['payment_installments'] == 0].shape[0]
payments['payment_installments'] = payments['payment_installments'].replace(0, 1)
print(f"  Fixed {invalid_before} invalid installment values → replaced with 1")


not_defined_before = payments[payments['payment_type'] == 'not_defined'].shape[0]
payments = payments[payments['payment_type'] != 'not_defined'].copy()
print(f"  Dropped {not_defined_before} rows with payment_type = 'not_defined'")

payments_agg = payments.groupby('order_id').agg(
    total_payment_value  = ('payment_value', 'sum'),
    payment_installments = ('payment_installments', 'max'),
    payment_type         = ('payment_type', 'first')
).reset_index()

print(f"\nSTEP 3 DONE: Payments cleaned and aggregated")
print(f"  Aggregated to {len(payments_agg):,} unique orders")
print(payments_agg.head(3))

  Fixed 2 invalid installment values → replaced with 1
  Dropped 3 rows with payment_type = 'not_defined'

STEP 3 DONE: Payments cleaned and aggregated
  Aggregated to 99,437 unique orders
                           order_id  total_payment_value  \
0  00010242fe8c5a6d1ba2dd792cb16214                72.19   
1  00018f77f2f0320c557190d7a144bdd3               259.83   
2  000229ec398224ef6ca0657da4fc703e               216.87   

   payment_installments payment_type  
0                     2  credit_card  
1                     3  credit_card  
2                     5  credit_card  


## Step 4 — Clean Products Table
Issues identified in extraction:
- 610 rows with missing product_category_name → fill with 'unknown' after joining translation
- 2 rows with missing weight and dimensions → drop only if needed for freight analysis
- Note: Olist dataset has a known typo — 'product_name_lenght' (missing 'h')
  We rename these columns to the correct spelling during cleaning.

In [6]:
products = products.rename(columns={
    'product_name_lenght'       : 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

products = products.merge(category_translation, on='product_category_name', how='left')

products['product_category_name_english'] = (
    products['product_category_name_english'].fillna('unknown')
)

for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    median_val = products[col].median()
    missing = products[col].isnull().sum()
    products[col] = products[col].fillna(median_val)
    print(f"  {col}: filled {missing} nulls with median ({median_val:.1f})")

print(f"\nSTEP 4 DONE: Products cleaned")
print(f"  Shape: {products.shape}")
print(products[['product_id','product_category_name_english','product_weight_g']].head(3))

  product_weight_g: filled 2 nulls with median (700.0)
  product_length_cm: filled 2 nulls with median (25.0)
  product_height_cm: filled 2 nulls with median (13.0)
  product_width_cm: filled 2 nulls with median (20.0)

STEP 4 DONE: Products cleaned
  Shape: (32951, 10)
                         product_id product_category_name_english  \
0  1e9e8ef04dbcff4541ed26657ea517e5                     perfumery   
1  3aa071139cb16b67ca9e5dea641aaa2f                           art   
2  96bd76ec8810374ed1b65e291975717f                sports_leisure   

   product_weight_g  
0             225.0  
1            1000.0  
2             154.0  


## Step 5 — Clean Customers & Sellers
Issues identified in extraction:
- Inconsistent city name casing in both tables → standardize to lowercase + strip whitespace

In [7]:
customers['customer_city'] = customers['customer_city'].str.lower().str.strip()
sellers['seller_city']     = sellers['seller_city'].str.lower().str.strip()

print("STEP 5 DONE: City names standardized to lowercase")
print(f"  Sample customer cities: {customers['customer_city'].unique()[:5].tolist()}")
print(f"  Sample seller cities  : {sellers['seller_city'].unique()[:5].tolist()}")

STEP 5 DONE: City names standardized to lowercase
  Sample customer cities: ['franca', 'sao bernardo do campo', 'sao paulo', 'mogi das cruzes', 'campinas']
  Sample seller cities  : ['campinas', 'mogi guacu', 'rio de janeiro', 'sao paulo', 'braganca paulista']


## Step 6 — Clean Geolocation Table
Issues identified in extraction:
- 261,831 duplicate rows (same zip code, multiple lat/lng entries)
- Solution: Deduplicate by taking mean lat/lng per zip code prefix

In [8]:
before = len(geolocation)

geo_clean = geolocation.groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat  = ('geolocation_lat', 'mean'),
    geolocation_lng  = ('geolocation_lng', 'mean'),
    geolocation_city = ('geolocation_city', 'first'),
    geolocation_state= ('geolocation_state', 'first')
).reset_index()

after = len(geo_clean)

print("STEP 6 DONE: Geolocation deduplicated")
print(f"  Before: {before:,} rows")
print(f"  After : {after:,} unique zip codes")
print(f"  Removed: {before - after:,} duplicate entries")

STEP 6 DONE: Geolocation deduplicated
  Before: 1,000,163 rows
  After : 19,015 unique zip codes
  Removed: 981,148 duplicate entries


## Step 7 — Clean Reviews Table
Issues identified in extraction:
- review_comment_title: 88.3% null → drop column
- review_comment_message: 58.7% null → drop column
- Keep only order_id and review_score for analysis
- Deduplicate to one review per order (keep first)

In [9]:
reviews_clean = reviews[['order_id', 'review_score']].copy()

dupes_before = reviews_clean.duplicated(subset='order_id').sum()
reviews_clean = reviews_clean.drop_duplicates(subset='order_id', keep='first')

print("STEP 7 DONE: Reviews cleaned")
print(f"  Dropped comment columns (88%+ null)")
print(f"  Removed {dupes_before} duplicate order reviews")
print(f"  Final shape: {reviews_clean.shape}")

STEP 7 DONE: Reviews cleaned
  Dropped comment columns (88%+ null)
  Removed 551 duplicate order reviews
  Final shape: (98673, 2)


## Step 8 — Aggregate Order Items
Each order can have multiple items (products).
Aggregating to one row per order: total price, total freight, item count.
Also keeping the first seller_id and product_id per order for joining.

In [10]:
order_items_agg = order_items.groupby('order_id').agg(
    total_items   = ('order_item_id', 'count'),
    total_price   = ('price', 'sum'),
    total_freight = ('freight_value', 'sum'),
    seller_id     = ('seller_id', 'first'),
    product_id    = ('product_id', 'first')
).reset_index()

print("STEP 8 DONE: Order items aggregated")
print(f"  Shape: {order_items_agg.shape}")
print(order_items_agg.head(3))

STEP 8 DONE: Order items aggregated
  Shape: (98666, 6)
                           order_id  total_items  total_price  total_freight  \
0  00010242fe8c5a6d1ba2dd792cb16214            1         58.9          13.29   
1  00018f77f2f0320c557190d7a144bdd3            1        239.9          19.93   
2  000229ec398224ef6ca0657da4fc703e            1        199.0          17.87   

                          seller_id                        product_id  
0  48436dade18ac8b2bce089ec2a041202  4244733e06e7ecb4970a6e2683c13e61  
1  dd7ddc04e1b6c2c614352b383efe2d36  e5f2d52b802189ee658865ca93d83a8f  
2  5b51032eddd242adc84c38acab88f23d  c777355d18b72b67abbeef9df44fd0fd  


## Step 9 — Merge All Tables into Master Dataset
Joining all cleaned tables on order_id to produce one unified analytical dataset.
Join sequence:
orders → customers → order_items → payments → reviews → products → sellers

In [11]:
master = orders_clean.merge(customers,       on='customer_id',  how='left')
master = master.merge(order_items_agg,       on='order_id',     how='left')
master = master.merge(payments_agg,          on='order_id',     how='left')
master = master.merge(reviews_clean,         on='order_id',     how='left')
master = master.merge(
    products[['product_id', 'product_category_name_english', 'product_weight_g']],
    on='product_id', how='left'
)
master = master.merge(
    sellers[['seller_id', 'seller_city', 'seller_state']],
    on='seller_id', how='left'
)

print("STEP 9 DONE: All tables merged into master dataset")
print(f"  Shape  : {master.shape}")
print(f"  Columns: {list(master.columns)}")

STEP 9 DONE: All tables merged into master dataset
  Shape  : (96478, 25)
  Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'total_items', 'total_price', 'total_freight', 'seller_id', 'product_id', 'total_payment_value', 'payment_installments', 'payment_type', 'review_score', 'product_category_name_english', 'product_weight_g', 'seller_city', 'seller_state']


In [12]:
assert len(master) == len(orders_clean), (
    f"Row mismatch! Expected {len(orders_clean):,}, got {len(master):,}"
)
print(f"✓ Join validation passed — {len(master):,} rows confirmed")

✓ Join validation passed — 96,478 rows confirmed


## Step 10 — Feature Engineering (Derived KPI Columns)
Creating derived features as defined in docs/data_dictionary.md — Planned Derived Features:
- delivery_time_days : actual time from purchase to delivery
- delivery_delay_days: how many days late or early vs estimate (positive = late)
- is_late_delivery   : boolean flag for late deliveries
- total_order_value  : price + freight per order
- customer_repeat_flag: True if customer placed more than one order
- order_month        : for monthly trend analysis
- order_year         : for yearly comparison

In [13]:
master['delivery_time_days'] = (
    master['order_delivered_customer_date'] - master['order_purchase_timestamp']
).dt.days

master['delivery_delay_days'] = (
    master['order_delivered_customer_date'] - master['order_estimated_delivery_date']
).dt.days

master['is_late_delivery'] = master['delivery_delay_days'] > 0

master['total_order_value'] = master['total_price'] + master['total_freight']

order_counts = master.groupby('customer_unique_id')['order_id'].count()
master['customer_repeat_flag'] = master['customer_unique_id'].map(order_counts > 1)

master['order_month'] = master['order_purchase_timestamp'].dt.to_period('M').astype(str)
master['order_year']  = master['order_purchase_timestamp'].dt.year

print("STEP 10 DONE: Derived features engineered")
print(master[[
    'delivery_time_days', 'delivery_delay_days',
    'is_late_delivery', 'total_order_value',
    'customer_repeat_flag', 'order_month'
]].head(5))

STEP 10 DONE: Derived features engineered
   delivery_time_days  delivery_delay_days  is_late_delivery  \
0                 8.0                 -8.0             False   
1                13.0                 -6.0             False   
2                 9.0                -18.0             False   
3                13.0                -13.0             False   
4                 2.0                -10.0             False   

   total_order_value  customer_repeat_flag order_month  
0              38.71                  True     2017-10  
1             141.46                 False     2018-07  
2             179.12                 False     2018-08  
3              72.20                 False     2017-11  
4              28.62                 False     2018-02  


## Step 11 — Final Null Check on Master Dataset
Verifying the merged dataset before saving.

In [14]:
print("=== FINAL NULL CHECK ON MASTER DATASET ===\n")
nulls = master.isnull().sum()
nulls = nulls[nulls > 0]

if nulls.empty:
    print("  No null values found in master dataset")
else:
    for col, count in nulls.items():
        pct = (count / len(master)) * 100
        print(f"  {col}: {count:,} nulls ({pct:.1f}%)")

print(f"\nMaster dataset shape: {master.shape}")
print(f"Total columns       : {master.shape[1]}")
print(f"Total rows          : {master.shape[0]:,}")

=== FINAL NULL CHECK ON MASTER DATASET ===



  order_approved_at: 14 nulls (0.0%)
  order_delivered_carrier_date: 2 nulls (0.0%)
  order_delivered_customer_date: 8 nulls (0.0%)
  total_payment_value: 1 nulls (0.0%)
  payment_installments: 1 nulls (0.0%)
  payment_type: 1 nulls (0.0%)
  review_score: 646 nulls (0.7%)
  delivery_time_days: 8 nulls (0.0%)
  delivery_delay_days: 8 nulls (0.0%)

Master dataset shape: (96478, 32)
Total columns       : 32
Total rows          : 96,478


In [15]:

median_score = master['review_score'].median()
master['review_score'] = master['review_score'].fillna(median_score)
print(f"  review_score: filled 646 nulls with median ({median_score})")

before = len(master)
master = master.dropna(subset=['order_delivered_customer_date'])
print(f"  Dropped {before - len(master)} rows with missing delivery timestamps")

master = master.dropna(subset=['total_payment_value'])
print(f"  Dropped 1 row with missing payment data")

print(f"\nFinal master shape after null treatment: {master.shape}")

remaining = master.isnull().sum().sum()
print(f"Total nulls remaining: {remaining}")

  review_score: filled 646 nulls with median (5.0)
  Dropped 8 rows with missing delivery timestamps


  Dropped 1 row with missing payment data

Final master shape after null treatment: (96469, 32)
Total nulls remaining: 15


### Final Validation — KPI Columns Check
Ensuring all critical analytical columns are free of null values before proceeding to EDA.

In [16]:
critical_cols = [
    'delivery_time_days',
    'delivery_delay_days',
    'total_order_value',
    'review_score',
    'customer_repeat_flag',
    'total_payment_value'
]

print("=== CRITICAL COLUMN NULL CHECK ===\n")
all_clean = True
for col in critical_cols:
    nulls = master[col].isnull().sum()
    status = "✓ CLEAN" if nulls == 0 else f"✗ {nulls} nulls"
    print(f"  {col}: {status}")
    if nulls > 0:
        all_clean = False

assert all_clean, "Critical KPI columns still have nulls!"
print("\n✓ All critical KPI columns are clean — safe to proceed to EDA")

=== CRITICAL COLUMN NULL CHECK ===

  delivery_time_days: ✓ CLEAN
  delivery_delay_days: ✓ CLEAN
  total_order_value: ✓ CLEAN
  review_score: ✓ CLEAN
  customer_repeat_flag: ✓ CLEAN
  total_payment_value: ✓ CLEAN

✓ All critical KPI columns are clean — safe to proceed to EDA


## Step 12 — Save Cleaned Master Dataset
Saving to data/processed/olist_master_cleaned.csv
This file will be used in all subsequent notebooks (EDA, statistical analysis, KPIs).

In [17]:
os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/olist_master_cleaned.csv"
master.to_csv(output_path, index=False)

print("STEP 12 DONE: Master dataset saved")
print(f"  Path : {output_path}")
print(f"  Shape: {master.shape}")
print(f"\nColumn list:")
for col in master.columns:
    print(f"  - {col}")

STEP 12 DONE: Master dataset saved
  Path : ../data/processed/olist_master_cleaned.csv
  Shape: (96469, 32)

Column list:
  - order_id
  - customer_id
  - order_status
  - order_purchase_timestamp
  - order_approved_at
  - order_delivered_carrier_date
  - order_delivered_customer_date
  - order_estimated_delivery_date
  - customer_unique_id
  - customer_zip_code_prefix
  - customer_city
  - customer_state
  - total_items
  - total_price
  - total_freight
  - seller_id
  - product_id
  - total_payment_value
  - payment_installments
  - payment_type
  - review_score
  - product_category_name_english
  - product_weight_g
  - seller_city
  - seller_state
  - delivery_time_days
  - delivery_delay_days
  - is_late_delivery
  - total_order_value
  - customer_repeat_flag
  - order_month
  - order_year


## ETL Pipeline Summary

| Step | Action | Result |
|---|---|---|
| 1 | Datetime conversion | 5 timestamp columns converted from string |
| 2 | Order filtering | Kept 96,478 delivered orders only |
| 3 | Payments cleaning | Fixed 0 installments, dropped not_defined, aggregated per order |
| 4 | Products cleaning | Fixed column typos, mapped English categories, imputed 2 null dimensions |
| 5 | City standardization | Lowercase + strip on customer and seller cities |
| 6 | Geolocation dedup | Mean lat/lng per zip — reduced from 1,000,163 to unique zip codes |
| 7 | Reviews cleaning | Kept review_score only, deduplicated per order |
| 8 | Order items aggregation | Summed price/freight, counted items per order |
| 9 | Master merge | All 9 tables joined on order_id |
| 10 | Feature engineering | delivery_time, delivery_delay, is_late, total_value, repeat_flag, order_month |
| 11 | Null check | Final verification before save |
| 12 | Export | Saved to data/processed/olist_master_cleaned.csv |
